In [1]:
import jax
jax.config.update('jax_num_cpu_devices', 8)
jax.devices()

[CpuDevice(id=0),
 CpuDevice(id=1),
 CpuDevice(id=2),
 CpuDevice(id=3),
 CpuDevice(id=4),
 CpuDevice(id=5),
 CpuDevice(id=6),
 CpuDevice(id=7)]

In [2]:
import jax.numpy as jnp
from jax import jit
import numpy as np

In [3]:
x = jnp.arange(5)
x.devices()
x.sharding
jax.debug.visualize_array_sharding(x)

  GPU 0  
         

In [4]:
import jax
print(jax.devices())

[CudaDevice(id=0)]


In [5]:
@jit
def f(x, neg):
  return -x if neg else x

f(1, True)

TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function f at /tmp/ipykernel_16916/2422663986.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument neg.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [6]:
from functools import partial

@partial(jit, static_argnums=(1,))
def f(x, neg):
  return -x if neg else x

f(1, True)

Array(-1, dtype=int32, weak_type=True)

In [4]:
@jit
def f(x):
  return x.reshape(np.array(x.shape).prod())

x = jnp.ones((2, 3))
np.array(x.shape).prod()

np.int64(6)

In [1]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
token = 'before'
stemmer.stem(token)

'befor'

In [ ]:
from nltk.corpus import stopwords as default_stopwords
default_stopwords.words("english")

In [22]:
x = jnp.arange(12).reshape((2, 6)) % 3
b = jnp.arange(5)

In [23]:
x

Array([[0, 1, 2, 0, 1, 2],
       [0, 1, 2, 0, 1, 2]], dtype=int32)

In [24]:
_eps = 1e-4
x = jnp.maximum(x, jnp.zeros_like(x))
norm = x.sum(axis=0)
jnp.where(norm > _eps, x / norm, jnp.zeros_like(x))


Array([[0. , 0.5, 0.5, 0. , 0.5, 0.5],
       [0. , 0.5, 0.5, 0. , 0.5, 0.5]], dtype=float32)

In [27]:
from cartm.preprocessing import DatasetPreprocessor
with open('./data/test_data.txt') as f:
    data = f.readlines()

preprocessor = DatasetPreprocessor(
    stopwords=set(), 
    min_word_len=0
    )
batch_data = preprocessor.fit_transform_batch(data, max_batch_size=20)
batch_data[0][0]

array([ 9,  8, 26, 28,  8, 27, 25, 18, 13, 24, 21,  2, 22,  7,  0, 13,  1,
       10, 14])

In [25]:
import jax
import jax.numpy as jnp
from functools import partial

@partial(jax.jit, static_argnames=['indices', 'gamma', 'beta'])
def _process_row(x: jax.Array, indices: jax.Array, gamma: float, beta: float) -> jax.Array:
    y = jnp.zeros_like(x)
    z = jnp.zeros_like(x)
    alpha = 1.0 - gamma
    starts = indices[:-1]
    ends = indices[1:]

    for k in range(len(starts)):
        s, e = int(starts[k]), int(ends[k])
        L = e - s
        if L <= 1:
            y = y.at[s:e].set(x[s:e])
            z = z.at[s:e].set(x[s:e])
            continue

        seg = x[s:e]

        # Прямая EMA: y_start = x_start, y_i = gamma*x_i + alpha*y_{i-1}
        def fwd(carry, val):
            res = gamma * val + alpha * carry
            return res, res
        _, y_rest = jax.lax.scan(fwd, seg[0], seg[1:], length=L-1)
        y = y.at[s:e].set(jnp.concatenate([seg[0:1], y_rest]))

        # Обратная EMA: z_end = x_end, z_{i-1} = gamma*x_{i-1} + alpha*z_i
        seg_rev = jnp.flip(seg)
        _, z_rest_rev = jax.lax.scan(fwd, seg_rev[0], seg_rev[1:], length=L-1)
        z = z.at[s:e].set(jnp.flip(jnp.concatenate([seg_rev[0:1], z_rest_rev])))

    return beta * y + (1.0 - beta) * z

# Батчирование по строкам (темам)
bidir_ema_jax = jax.vmap(_process_row, in_axes=(0, None, None, None))

In [28]:
bidir_ema_jax(x=batch_data[0][0], indices=batch_data[0][1], gamma=0.6, beta=0.5)

ValueError: vmap in_axes must be an int, None, or a tuple of entries corresponding to the positional arguments passed to the function, but got len(in_axes)=4, len(args)=0

In [35]:
import jax
import jax.numpy as jnp
from functools import partial
import time

# ---------------------------------------------------------
# 1. Определяем функцию (код из предыдущего сообщения)
# ---------------------------------------------------------
@partial(jax.jit, static_argnames=['indices'])
def _process_row(x: jax.Array, indices, gamma: float, beta: float) -> jax.Array:
    y = jnp.zeros_like(x)
    z = jnp.zeros_like(x)
    alpha = 1.0 - gamma
    starts = indices[:-1]
    ends = indices[1:]

    for k in range(len(starts)):
        s, e = int(starts[k]), int(ends[k])
        L = e - s
        if L <= 1:
            y = y.at[s:e].set(x[s:e])
            z = z.at[s:e].set(x[s:e])
            continue

        seg = x[s:e]

        def fwd(carry, val):
            res = gamma * val + alpha * carry
            return res, res
        _, y_rest = jax.lax.scan(fwd, seg[0], seg[1:], length=L-1)
        y = y.at[s:e].set(jnp.concatenate([seg[0:1], y_rest]))

        seg_rev = jnp.flip(seg)
        _, z_rest_rev = jax.lax.scan(fwd, seg_rev[0], seg_rev[1:], length=L-1)
        z = z.at[s:e].set(jnp.flip(jnp.concatenate([seg_rev[0:1], z_rest_rev])))

    return beta * y + (1.0 - beta) * z

bidir_ema_jax = jax.vmap(_process_row, in_axes=(0, None, None, None))

# ---------------------------------------------------------
# 2. Подготовка данных
# ---------------------------------------------------------
H, I = 3, 12  # 3 темы, 12 позиций в каждой строке
X = jnp.array([
    [0.1, 0.5, 0.9, 0.2, 0.3, 0.8, 0.4, 0.1, 0.7, 0.6, 0.2, 0.9],
    [0.8, 0.2, 0.1, 0.6, 0.9, 0.3, 0.5, 0.1, 0.4, 0.7, 0.8, 0.3],
    [0.3, 0.4, 0.2, 0.9, 0.1, 0.6, 0.8, 0.5, 0.2, 0.3, 0.7, 0.1]
], dtype=jnp.float32)

# Границы документов: 3 сегмента [0:4], [4:8], [8:12]
# Передаём как tuple/list, т.к. indices указан в static_argnames
indices = (0, 4, 8, 12)

gamma = 0.6
beta  = 0.5

# ---------------------------------------------------------
# 3. Вызов и замер времени
# ---------------------------------------------------------
# Первый вызов включает компиляцию XLA (займёт 0.1–0.5 сек)
start = time.time()
result = bidir_ema_jax(X, indices, gamma, beta)
result.block_until_ready()  # Синхронизация для GPU/TPU
print(f"Первый вызов (с компиляцией): {time.time()-start:.4f} сек")

# Последующие вызовы используют кэшированное ядро
start = time.time()
for _ in range(10):
    bidir_ema_jax(X, indices, gamma, beta).block_until_ready()
print(f"10 последующих вызовов: {(time.time()-start)/10:.4f} сек/итерация")

print("\nФорма результата:", result.shape)
print(result)

Первый вызов (с компиляцией): 0.4317 сек
10 последующих вызовов: 0.0009 сек/итерация

Форма результата: (3, 12)
[[0.18959999 0.444      0.648      0.2952     0.3584     0.596
  0.38       0.176      0.6704     0.59599996 0.42799997 0.7952    ]
 [0.688      0.34       0.268      0.5272     0.7832     0.42799997
  0.42799997 0.18319999 0.45200002 0.62       0.656      0.3824    ]
 [0.3264     0.396      0.37199998 0.77279997 0.20639998 0.516
  0.65999997 0.528      0.2328     0.31199998 0.49199998 0.18479998]]


In [36]:
import jax
import jax.numpy as jnp
from functools import partial

@partial(jax.jit, static_argnames=['indices'])
def _process_row(x, indices, gamma, beta):
    y = jnp.zeros_like(x)
    z = jnp.zeros_like(x)
    alpha = 1.0 - gamma
    for k in range(len(indices) - 1):
        s, e = int(indices[k]), int(indices[k+1])
        L = e - s
        if L <= 1:
            y = y.at[s:e].set(x[s:e])
            z = z.at[s:e].set(x[s:e])
            continue
        seg = x[s:e]
        _, y_rest = jax.lax.scan(lambda c, v: (gamma * v + alpha * c, gamma * v + alpha * c), seg[0], seg[1:], length=L-1)
        y = y.at[s:e].set(jnp.concatenate([seg[0:1], y_rest]))
        seg_rev = jnp.flip(seg)
        _, z_rev = jax.lax.scan(lambda c, v: (gamma * v + alpha * c, gamma * v + alpha * c), seg_rev[0], seg_rev[1:], length=L-1)
        z = z.at[s:e].set(jnp.flip(jnp.concatenate([seg_rev[0:1], z_rev])))
    return beta * y + (1.0 - beta) * z

bidir_ema_jax = jax.vmap(_process_row, in_axes=(0, None, None, None))

# --- Использование ---
X = jnp.ones((2, 6), dtype=jnp.float32)
indices = (0, 3, 6)          # tuple/list, не jnp.array
gamma, beta = 0.6, 0.5       # обычные float

out = bidir_ema_jax(X, indices, gamma, beta)
out.block_until_ready()
print(out.shape)  # (2, 6)

(2, 6)


In [37]:
out

Array([[1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1.]], dtype=float32)